In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [ ]:
llm = ChatOllama(
    model="qwen3.8:27b",
    base_url="http://192.168.24.150:11434",
    temperature=0,
)

# invoke — 기본 질의응답
- 문자열을 넣으면 → 자동으로 `HumanMessage`로 변환
- 메시지 리스트를 넣으면 → system 역할까지 지정 가능

In [ ]:
# 방법 1: 문자열로 간단히 질문
response = llm.invoke("LangChain이 무엇인지 두 문장으로 설명해 주세요.")
print(type(response).__name__)   # AIMessage
print(response.content)

In [ ]:
# 방법 2: 메시지 리스트로 system 역할 지정
messages = [
    SystemMessage(content="당신은 모든 답을 반말로 짧게 하는 친구입니다."),
    HumanMessage(content="파이썬 배우면 뭐가 좋아?"),
]
response = llm.invoke(messages)
print(response.content)

# batch — 여러 질문 한 번에 처리

`batch()`는 리스트의 각 항목에 대해 내부적으로 `_generate()`를 호출합니다.

> ⚠️ **로컬 GPU 모델 주의점**: LangChain의 `batch()`는 기본적으로 **여러 스레드로 동시 실행**을
> 시도합니다. API 서버라면 빨라지지만, GPU가 1개뿐인 로컬 모델은 동시 호출 시 충돌/속도 저하가
> 생길 수 있습니다. → `max_concurrency=1`로 **순차 실행**을 지정하는 것이 안전합니다.


In [ ]:
questions = [
    "지구에서 가장 큰 바다는?",
    "1 + 1 은 왜 2인가요? 한 문장으로.",
    "김밥의 주재료 3가지만 알려주세요.",
]

# 로컬 GPU 1개 → 순차 실행이 안전
answers = llm.batch(questions, config={"max_concurrency": 3})

for q, a in zip(questions, answers):
    print(f"Q: {q}")
    print(f"A: {a.content}")
    print("-" * 60)


# stream — 생성되는 텍스트를 실시간으로 출력

`model.generate()`는 응답 생성이 끝날 때까지 기다리는 함수입니다.
따라서 백그라운드 스레드에서 모델을 실행하고,
메인 스레드에서는 `TextIteratorStreamer`가 전달하는 텍스트를 출력합니다.

Jupyter에서 바로 표시되도록 `flush=True`를 사용합니다.

In [ ]:
for chunk in llm.stream("가을에 대한 시를 한 편 지어주세요."):
    print(chunk.content, end="", flush=True)

In [ ]:
for chunk in llm.stream("gitea vs gitlab 차이점을 설명해 주세요."):
    print(chunk.content, end="", flush=True)